# 검증 세트

태스트 세트를 사용하지 않으면 모델이 과대적합인지 과소적합인지 판단하기 어려움

테스트 세트를 사용하지 않고 이를 측정하는 간단한 방법은 훈련 세트를 나누는거임

이 데이터를 검증세트라고함

훈련세트로 모델을 훈련 -> 검증 세트로 모델을 평가 -> 가장 좋은 모델을 고름 -> 이 매개변수를 사용해 훈련 세트와 검증 세트를 합쳐 훈련 데이터에서 모델 다시 훈련 -> 테스트 세트에서 최종 점수 평가

In [1]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [2]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [3]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)
#0.2는 20%만 테스트 세트로 나눈다.

In [4]:
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42)
#train_input의 약 20%fmf val_input으로 만듦

In [5]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [6]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))
#과대적합, 매개변수를 바꿔서 더 좋은 모델을 찾아야함

0.9971133028626413
0.864423076923077


# 교차 검증

검증 세트를 만드느라 훈련 세트가 줄었음


보통 많은 데이터를 훈련하는것이 좋은 모델로 만들어짐

검증 세트를 너무 조금 떼어 놓으면 검증 점수가 불안정함
-> 교차검증을 이용



교차 검증은 검증 세트를 떼어 내어 평가하는 과정을 여러 번 반복함. 그 다음 이 점수를 평균하여 최종 검증 점수를 얻음.

훈련 세트를 몇 부분으로 나누냐에 따라 다르게 부름 (k-겹 교차 검증)


In [7]:
from sklearn.model_selection import cross_validate
scores=cross_validate(dt,train_input,train_target)
#교차 검증 함수 (평가할 모델 객체,검증세트 때어내지 않고 훈련 세트)
#기본적으로 5-폴드 교차 검증
print(scores)

{'fit_time': array([0.01134133, 0.00908852, 0.00931215, 0.01068568, 0.01021075]), 'score_time': array([0.00165296, 0.00152922, 0.00151348, 0.00282788, 0.00216603]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [8]:
import numpy as np
print(np.mean(scores['test_score']))

0.855300214703487


In [10]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))
#교차 검증을 할 때 훈련 세트를 섞으려면 분할기를 지정해야함
#cross_validate()함수는 기본적으로 회귀 모델일 경우 K-Fold분할기를 사용
#분류 모델일 경우 타깃 클래스르 골고루 나누기 위해 StratfiedKFold를 사용합니다.


0.855300214703487


In [11]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores=cross_validate(dt,train_input,train_target,cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


결정 트리의 매개변수 값을 바꿔가며 가장 좋은 성능이 나오는 모델을 찾기

# 하이퍼파라미터 튜닝

사용자가 지정해야만 하는 파라미터를 하이퍼파라미터라고 함

1. 라이브러리가 제공하는 기본값을 그대로 사용해 훈련
2. 검증 세트의 점수나 교차 검증을 통해서 매개변수를 조금씩 바꿈
3. 매개변수를 바꿔가면서 모델을 훈련하고 교차 검증

결정트리에서 max_depth의 최적값은 min_samples_split 매개변수 값이 바뀌면 달라지니까
두 매개변수를 동시에 바꿔가며 최적의 값을 찾아야함

매개변수가 많아지면 문제가 더 복잡해짐
**그리드 서치**를 사용함




In [22]:

from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.00001, 0.00002, 0.00003, 0.00004, 0.00005,0.00006,0.00007,0.00008,0.00009,0.0001,0.00011]}

#별도로 cross_validate()함수를 호출할 필요가 없음


In [23]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params,n_jobs=-1)
#근데 5-폴드 교차 검증하니까 10x5=50개의 모델을 훈련함
#의 n_jobs 매개변수에서 병렬 실행에 사용할 CPU 코어 수를 지정하는 것이 좋습니다.
#이 매개변수의 기본값은 1입니다. -1로 지정하면 시스템에 있는 모든 코어를 사용합니다.


In [24]:
gs.fit(train_input,train_target)
#25개의 모델 중에서 검증 점스가 가장 높은 모델의 매개변수 조합으로
#전체 훈련 세트에서 자동으로 다시 모델을 훈련합니다.

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [1e-05, 2e-05, 3e-05, 4e-05,
                                                   5e-05, 6e-05, 7e-05, 8e-05,
                                                   9e-05, 0.0001, 0.00011]})

In [25]:
dt=gs.best_estimator_
print(dt.score(train_input,train_target))

0.9615162593804117


In [26]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [27]:
print(gs.cv_results_['mean_test_score'])
#각 매개변수에서 수행한 교차 검증의 평균 점수

[0.85664711 0.85606963 0.8593396  0.86145721 0.86126527 0.86107315
 0.86511346 0.8647294  0.86530762 0.86819297 0.86703783]


In [28]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [32]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.00005),
          'max_depth': range(3, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }
#첫번째 매개변수 9개
#두번째 매개변수 15개
#세번째 메개뱐수 10개 9*15*10*5

In [33]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(3, 20),
                         'min_impurity_decrease': array([0.0001 , 0.00015, 0.0002 , 0.00025, 0.0003 , 0.00035, 0.0004 ,
       0.00045, 0.0005 , 0.00055, 0.0006 , 0.00065, 0.0007 , 0.00075,
       0.0008 , 0.00085, 0.0009 , 0.00095]),
                         'min_samples_split': range(2, 100, 10)})

In [34]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.00045000000000000004), 'min_samples_split': 12}


In [35]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8697336566224921


# 랜덤 서치

매개 변수의 값이 수치일 때 값의 범위나 간격을 미리 정하기 어려울 수 있음

너무 많은 매개변수 조건이 있어 그리드 서치 수행 시간이 오래 걸릴 수 있음

-> 랜덤 서치 사용

매개변수 값의 목록을 전달하는 것이 아니라 매개변수를 샘플링할수 있는 확률 분포 객체를 전달

In [39]:
from scipy.stats import uniform, randint

In [ ]:
rgen=randint(0,10)
rgen.rvs(10)

array([2, 8, 8, 6, 1, 6, 8, 6, 9, 7])

In [ ]:
np.unique(rgen.rvs(1000),return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([114,  98,  84, 106, 100,  92, 107,  93,  99, 107]))

In [ ]:
ugen=uniform(0,1)
ugen.rvs(10)

array([0.3009037 , 0.8722084 , 0.40749032, 0.71865657, 0.69723267,
       0.76837745, 0.64536828, 0.19646807, 0.04047969, 0.69518233])

In [40]:
params = {'min_impurity_decrease': uniform(0.00001, 0.001),
          'max_depth': randint(3, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25),
          }

In [41]:
from sklearn.model_selection import RandomizedSearchCV
gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params,
n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)
#n_iter:params에 정의된 매개변수 범위에서 총 100번을 샘플링하여 교차 검증을 수행하고 최적의 매개변수 조합을 찾음

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ec99616a630>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7ec995496270>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ec995495610>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7ec995c6af60>},
                   random_state=42)

In [43]:
print(gs.best_params_)

{'max_depth': 29, 'min_impurity_decrease': np.float64(0.000427411003148779), 'min_samples_leaf': 5, 'min_samples_split': 2}


In [44]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8697336566224921


In [45]:
dt = gs.best_estimator_#여기에 저장되어있음
print(dt.score(test_input, test_target))

0.8592307692307692
